# 💬 Chat with Your Fine-Tuned Model (Research Grade v2)

**Key fixes in this version:**
- ✅ System prompt loaded from `training_config.json` (must match training)
- ✅ Inference decodes only NEW tokens — prompt no longer leaks into response
- ✅ All broken `split('### Assistant:')` parsing removed
- ✅ `repetition_penalty=1.15` prevents hallucination loops
- ✅ Conversation history properly managed

**Requirements:**
- Your trained model in Google Drive (`Finetune_Jobs/models/your-model-name/`)
- T4 GPU (Runtime → Change runtime type → T4 GPU)

**Time:** ~3 minutes to load, then instant responses

## ⚙️ Step 1: Configuration

In [ ]:
# ============================================================================
# CONFIGURATION - UPDATE THIS
# ============================================================================

# Your model name — must match exactly what you used in Fine_Tune_Llama3.ipynb
MODEL_NAME = "customer-support-bot-v1"  # <- CHANGE THIS

# Model path in Google Drive
MODEL_PATH = f"/content/drive/MyDrive/Finetune_Jobs/models/{MODEL_NAME}"

# Chat settings
TEMPERATURE        = 0.7   # 0.0 = deterministic, 1.0 = creative
MAX_TOKENS         = 256   # Maximum new tokens per response
TOP_P              = 0.9   # Nucleus sampling
REPETITION_PENALTY = 1.15  # >1.0 penalises repeated tokens — reduces hallucination loops

# System prompt — will be auto-loaded from training_config.json
# Only used as fallback if config file is not found
FALLBACK_SYSTEM_PROMPT = (
    "You are a helpful, accurate, and concise AI assistant. "
    "Answer questions clearly based on what you know with confidence. "
    "If you are unsure about something, say so honestly rather than guessing."
)

print("✅ Configuration loaded")
print(f"   Model     : {MODEL_NAME}")
print(f"   Path      : {MODEL_PATH}")
print(f"   Temp      : {TEMPERATURE}")
print(f"   Max tokens: {MAX_TOKENS}")

## 🔗 Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os
import json

drive.mount('/content/drive')

# Verify model exists
if not os.path.exists(MODEL_PATH):
    print(f"❌ ERROR: Model not found at {MODEL_PATH}")
    models_dir = "/content/drive/MyDrive/Finetune_Jobs/models"
    if os.path.exists(models_dir):
        available = os.listdir(models_dir)
        print(f"\nAvailable models in Finetune_Jobs/models/:")
        for m in available:
            print(f"  - {m}")
    raise FileNotFoundError("Please update MODEL_NAME in Step 1")

print(f"✅ Drive mounted")
print(f"✅ Model found: {MODEL_PATH}")

# List model files
print(f"\n📁 Model files:")
for f in os.listdir(MODEL_PATH):
    print(f"   - {f}")

# Load system prompt from training config (CRITICAL — must match training)
config_path = os.path.join(MODEL_PATH, "training_config.json")
if os.path.exists(config_path):
    with open(config_path) as f:
        training_config = json.load(f)
    SYSTEM_PROMPT = training_config.get("system_prompt", FALLBACK_SYSTEM_PROMPT)
    print(f"\n✅ System prompt loaded from training_config.json")
    print(f"   System prompt: \"{SYSTEM_PROMPT[:80]}...\"")
else:
    SYSTEM_PROMPT = FALLBACK_SYSTEM_PROMPT
    print(f"\n⚠️  training_config.json not found — using fallback system prompt")
    print(f"   Make sure this matches what you used during training!")
    print(f"   System prompt: \"{SYSTEM_PROMPT[:80]}...\"") 

## 📦 Step 3: Install Unsloth

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

print("✅ Unsloth installed")

## 🤖 Step 4: Load Your Model

**This takes ~2-3 minutes. Please wait...**

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

print("⏳ Loading base model (Llama 3 8B 4-bit)...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

print("⏳ Loading your fine-tuned LoRA adapter...")

model = PeftModel.from_pretrained(model, MODEL_PATH)

# Enable Unsloth's optimised inference mode
FastLanguageModel.for_inference(model)

print("✅ Model loaded and ready!")
print(f"   Model        : {MODEL_NAME}")
print(f"   Fast inference: enabled")
print(f"   System prompt : loaded and will be used on every turn")

## 💬 Step 5: Chat Function

**Key fixes vs original:**
- System prompt prepended to every call (matches training format)
- `input_len` tracked so only new tokens are decoded — no more prompt leakage
- Removed broken `split('### Assistant:')` parsing entirely
- `repetition_penalty` added to reduce hallucination loops

In [ ]:
def chat(
    message,
    history            = None,
    temperature        = TEMPERATURE,
    max_tokens         = MAX_TOKENS,
    top_p              = TOP_P,
    repetition_penalty = REPETITION_PENALTY
):
    """
    Chat with your fine-tuned model.

    Args:
        message            : User's input message
        history            : List of previous {role, content} dicts (optional)
        temperature        : 0.0 = deterministic, 1.0 = creative
        max_tokens         : Max new tokens to generate
        top_p              : Nucleus sampling probability
        repetition_penalty : >1.0 reduces repetitive outputs (1.15 is a safe default)

    Returns:
        str: The model's response (clean, no prompt text)
    """
    if history is None:
        history = []

    # Build full message list:
    # system prompt first -> conversation history -> current user message
    # CRITICAL: system prompt MUST be first, and MUST match training exactly
    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + history
        + [{"role": "user", "content": message}]
    )

    # Tokenize using Llama 3 chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,   # Adds the assistant turn opener
        return_tensors        = "pt"
    ).to("cuda")

    # Record prompt length so we can slice it away from the output
    input_len = inputs.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids          = inputs,
            max_new_tokens     = max_tokens,
            temperature        = temperature,
            top_p              = top_p,
            do_sample          = True,
            repetition_penalty = repetition_penalty,  # Prevents hallucination loops
            pad_token_id       = tokenizer.eos_token_id
        )

    # CRITICAL FIX: Decode ONLY the newly generated tokens.
    # The original notebook decoded outputs[0] (entire sequence including prompt),
    # then tried to split on '### Assistant:' which does not exist in Llama 3
    # ChatML format. This caused the full prompt to appear in responses.
    new_tokens = outputs[0][input_len:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return response.strip()


print("✅ Chat function ready")
print("   Usage: chat('your message')")
print("   Multi-turn: chat('message', history=conversation_history)")

## 🧪 Step 6: Quick Test

In [ ]:
import time

test_message = "Hello! Can you help me?"

print(f"👤 User: {test_message}")
print("\n⏳ Generating...\n")

t0       = time.time()
response = chat(test_message)
elapsed  = time.time() - t0

print(f"🤖 {MODEL_NAME}: {response}")
print(f"\n⏱️  Response time: {elapsed:.2f}s")
print("\n✅ Model is working correctly!")
print("   Verify: the response should NOT contain the system prompt or user message.")
print("   If you see prompt text in the output, the model path may be wrong.")

## 💬 Step 7: Interactive Multi-Turn Chat

Run this cell and start chatting. Maintains full conversation history.
Type `quit`, `exit`, or `bye` to stop. Type `reset` to start a new conversation.

In [ ]:
import time

print("=" * 70)
print(f"💬 Chat with {MODEL_NAME}")
print("=" * 70)
print("Commands: 'quit'/'exit' to stop | 'reset' to clear history | 'history' to view")
print(f"System prompt: \"{SYSTEM_PROMPT[:60]}...\"")
print("=" * 70)

conversation_history = []   # Stores {role, content} dicts for multi-turn context
MAX_HISTORY_TURNS    = 5    # Keep last N turns to avoid context overflow on T4

while True:
    user_message = input("\n👤 You: ").strip()

    # --- Commands ---
    if user_message.lower() in ['quit', 'exit', 'bye']:
        print("\n👋 Goodbye!")
        break

    if user_message.lower() == 'reset':
        conversation_history = []
        print("   🔄 Conversation history cleared.")
        continue

    if user_message.lower() == 'history':
        if not conversation_history:
            print("   (No history yet)")
        for h in conversation_history:
            print(f"   [{h['role']:9s}] {h['content'][:80]}")
        continue

    if not user_message:
        continue

    # --- Generate response ---
    t0       = time.time()
    response = chat(user_message, history=conversation_history)
    elapsed  = time.time() - t0

    print(f"\n🤖 {MODEL_NAME}: {response}")
    print(f"   ⏱️  {elapsed:.2f}s | history: {len(conversation_history)//2} turns")

    # Update conversation history
    conversation_history.append({"role": "user",      "content": user_message})
    conversation_history.append({"role": "assistant", "content": response})

    # Trim history to avoid GPU OOM on T4
    # Keep only the last MAX_HISTORY_TURNS exchanges (each exchange = 2 entries)
    if len(conversation_history) > MAX_HISTORY_TURNS * 2:
        conversation_history = conversation_history[-(MAX_HISTORY_TURNS * 2):]

## 🌡️ Step 8: Temperature Comparison

Same question at 3 temperature settings — understand how temperature affects your model.

In [ ]:
question = "What is machine learning?"  # <- Change to any question

print(f"Question: \"{question}\"")
print("=" * 70)

for temp in [0.1, 0.5, 0.9]:
    label = {0.1: "Focused (deterministic)", 0.5: "Balanced", 0.9: "Creative"}[temp]
    print(f"\n🌡️  Temperature {temp} — {label}")
    response = chat(question, temperature=temp)
    print(f"🤖 {response}")
    print("-" * 70)

## 📊 Step 9: Batch Test (Multiple Questions)

Test your model on a list of questions at once. Good for systematic evaluation.

In [ ]:
import time

# Edit these to match your domain
TEST_QUESTIONS = [
    "What is machine learning?",
    "How can I reset my password?",
    "Tell me about neural networks",
    "What's the difference between AI and ML?",
    "I'm not sure what to ask — can you suggest something?",
]

print(f"🧪 Batch test — {len(TEST_QUESTIONS)} questions\n")
print("=" * 70)

timings = []

for i, question in enumerate(TEST_QUESTIONS, 1):
    t0       = time.time()
    response = chat(question)
    elapsed  = time.time() - t0
    timings.append(elapsed)

    print(f"\n{i}. 👤 {question}")
    print(f"   🤖 {response}")
    print(f"   ⏱️  {elapsed:.2f}s")
    print("-" * 70)

avg_time = sum(timings) / len(timings)
print(f"\n📊 Average response time: {avg_time:.2f}s")

## 🔍 Step 10: CUP Hallucination Check

Ask each question 3 ways and measure answer consistency.
CUP > 0.6 = consistent ✅ | CUP < 0.3 = possibly hallucinating ❌

In [ ]:
def cup_check(questions):
    """
    Consistency Under Paraphrase check.
    Uses greedy decoding (temperature=0.1, do_sample=False) for fair comparison.
    """
    def word_overlap(a, b):
        sa = set(a.lower().split())
        sb = set(b.lower().split())
        return len(sa & sb) / max(len(sa | sb), 1)

    all_cups = []

    for q in questions:
        variants = [
            q,
            f"Can you explain: {q}",
            f"I'd like to understand: {q}"
        ]
        answers = [chat(v, temperature=0.1) for v in variants]

        pairs  = [(0,1), (0,2), (1,2)]
        scores = [word_overlap(answers[i], answers[j]) for i, j in pairs]
        cup    = sum(scores) / 3
        all_cups.append(cup)

        flag = "✅" if cup >= 0.6 else ("⚠️" if cup >= 0.3 else "❌")
        print(f"{flag} CUP={cup:.3f}  |  Q: {q}")
        for vi, (variant, answer) in enumerate(zip(variants, answers)):
            print(f"   V{vi+1}: {answer[:80]}")
        print()

    avg = sum(all_cups) / max(len(all_cups), 1)
    print(f"Average CUP Score: {avg:.3f}")
    if avg >= 0.6:
        print("✅ Model is CONSISTENT — low hallucination risk")
    elif avg >= 0.3:
        print("⚠️  MODERATELY consistent — some hallucination risk")
    else:
        print("❌ INCONSISTENT — high hallucination risk")
    return avg


# Edit these to match your domain
PROBE_QUESTIONS = [
    "What is machine learning?",
    "How does gradient descent work?",
    "What is a neural network?"
]

print("🔍 Running CUP hallucination check...\n")
avg_cup = cup_check(PROBE_QUESTIONS)

## 💾 Step 11: Save Conversation to Google Drive (Optional)

In [ ]:
import json
import os
from datetime import datetime

if conversation_history:
    timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_dir   = "/content/drive/MyDrive/Finetune_Jobs/conversations"
    save_path  = f"{save_dir}/chat_{MODEL_NAME}_{timestamp}.json"

    os.makedirs(save_dir, exist_ok=True)

    payload = {
        "model"        : MODEL_NAME,
        "system_prompt": SYSTEM_PROMPT,
        "timestamp"    : timestamp,
        "turns"        : len(conversation_history) // 2,
        "conversation" : conversation_history
    }

    with open(save_path, 'w') as f:
        json.dump(payload, f, indent=2)

    print(f"✅ Conversation saved to: {save_path}")
    print(f"   Turns saved: {len(conversation_history) // 2}")
else:
    print("ℹ️  No conversation to save. Use Step 7 to chat first.")